In [1]:
# ============================================
# Import libraries
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, regexp_replace
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import udf

from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

In [2]:
# ============================================
# Create Spark session
# ============================================

spark = SparkSession.builder \
    .appName("Product Name Clustering") \
    .getOrCreate()

In [3]:
# ============================================
# Define products schema and load file
# ============================================

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])

products = spark.read.csv(
    "products.csv",
    header=True,
    schema=products_schema
)

products.show(5, False)
products.printSchema()

+----------+-----------------------------------------------------------------+--------+-------------+
|product_id|product_name                                                     |aisle_id|department_id|
+----------+-----------------------------------------------------------------+--------+-------------+
|1         |Chocolate Sandwich Cookies                                       |61      |19           |
|2         |All-Seasons Salt                                                 |104     |13           |
|3         |Robust Golden Unsweetened Oolong Tea                             |94      |7            |
|4         |Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce|38      |1            |
|5         |Green Chile Anytime Sauce                                        |5       |13           |
+----------+-----------------------------------------------------------------+--------+-------------+
only showing top 5 rows
root
 |-- product_id: integer (nullable = true)
 |-- produ

In [4]:
# ============================================
# Clean and normalize product names
# ============================================

products_clean = products.select("product_id", "product_name") \
    .dropna(subset=["product_name"]) \
    .dropDuplicates(["product_id"])

products_clean = products_clean.withColumn(
    "product_name_clean",
    trim(
        regexp_replace(
            lower(col("product_name")),
            "[^a-zA-Z0-9 ]",
            ""
        )
    )
)

products_clean.select("product_name", "product_name_clean").show(10, False)

+-----------------------------------------------------------------+-----------------------------------------------------------------+
|product_name                                                     |product_name_clean                                               |
+-----------------------------------------------------------------+-----------------------------------------------------------------+
|Chocolate Sandwich Cookies                                       |chocolate sandwich cookies                                       |
|All-Seasons Salt                                                 |allseasons salt                                                  |
|Robust Golden Unsweetened Oolong Tea                             |robust golden unsweetened oolong tea                             |
|Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce|smart ones classic favorites mini rigatoni with vodka cream sauce|
|Green Chile Anytime Sauce                                    

In [5]:
# ============================================
# Text vectorization and clustering pipeline
# ============================================

tokenizer = Tokenizer(
    inputCol="product_name_clean",
    outputCol="words"
)

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

count_vectorizer = CountVectorizer(
    inputCol="filtered_words",
    outputCol="raw_features",
    vocabSize=5000,
    minDF=5
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

kmeans = KMeans(
    featuresCol="features",
    predictionCol="product_cluster",
    k=12,
    seed=42
)

pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    count_vectorizer,
    idf,
    kmeans
])

model = pipeline.fit(products_clean)

product_clusters = model.transform(products_clean)

In [6]:
# ============================================
# View clustering results
# ============================================

product_clusters.select(
    "product_id",
    "product_name",
    "product_name_clean",
    "product_cluster"
).show(30, False)

+----------+-----------------------------------------------------------------+-----------------------------------------------------------------+---------------+
|product_id|product_name                                                     |product_name_clean                                               |product_cluster|
+----------+-----------------------------------------------------------------+-----------------------------------------------------------------+---------------+
|1         |Chocolate Sandwich Cookies                                       |chocolate sandwich cookies                                       |11             |
|2         |All-Seasons Salt                                                 |allseasons salt                                                  |0              |
|3         |Robust Golden Unsweetened Oolong Tea                             |robust golden unsweetened oolong tea                             |0              |
|4         |Smart Ones Classic Fav

In [7]:
# ============================================
# Count products in each cluster
# ============================================

product_clusters.groupBy("product_cluster") \
    .count() \
    .orderBy("product_cluster") \
    .show()

+---------------+-----+
|product_cluster|count|
+---------------+-----+
|              0|37297|
|              1|  981|
|              2| 1432|
|              3| 3052|
|              4|  606|
|              5| 1648|
|              6|  664|
|              7|   27|
|              8|  172|
|              9|  558|
|             10|  902|
|             11| 2349|
+---------------+-----+



In [8]:
# ============================================
# Inspect examples from each cluster
# ============================================

for i in range(12):
    print(f"\n===== Cluster {i} =====")
    product_clusters.filter(col("product_cluster") == i) \
        .select("product_name") \
        .show(15, False)


===== Cluster 0 =====
+-----------------------------------------------------------------+
|product_name                                                     |
+-----------------------------------------------------------------+
|All-Seasons Salt                                                 |
|Robust Golden Unsweetened Oolong Tea                             |
|Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce|
|Green Chile Anytime Sauce                                        |
|Dry Nose Oil                                                     |
|Pure Coconut Water With Orange                                   |
|Cut Russet Potatoes Steam N' Mash                                |
|Sparkling Orange Juice & Prickly Pear Beverage                   |
|Peach Mango Juice                                                |
|Saline Nasal Mist                                                |
|Fresh Scent Dishwasher Cleaner                                   |
|Overnight Diapers Size 6

In [10]:
cluster_names = {
    0: "General Grocery & Drinks",
    1: "Turkey & Roasted Foods",
    2: "Vanilla & Yogurt Products",
    3: "Bread, Chips & Grain Snacks",
    4: "Baby & Organic Baby Products",
    5: "Cheese Products",
    6: "Cheddar & Dairy Snacks",
    7: "Rings & Snack Foods",
    8: "Face & Skin Care",
    9: "Greek Yogurt Products",
    10: "Fruit Snacks & Fruit Drinks",
    11: "Chocolate Products"
}

In [12]:
def map_cluster_name(cluster_id):
    return cluster_names.get(cluster_id, "Other")

cluster_name_udf = udf(
    map_cluster_name,
    StringType()
)

product_clusters = product_clusters.withColumn(
    "cluster_name",
    cluster_name_udf(col("product_cluster"))
)

In [13]:
product_clusters.select(
    "product_name",
    "product_cluster",
    "cluster_name"
).show(30, False)

+-----------------------------------------------------------------+---------------+---------------------------+
|product_name                                                     |product_cluster|cluster_name               |
+-----------------------------------------------------------------+---------------+---------------------------+
|Chocolate Sandwich Cookies                                       |11             |Chocolate Products         |
|All-Seasons Salt                                                 |0              |General Grocery & Drinks   |
|Robust Golden Unsweetened Oolong Tea                             |0              |General Grocery & Drinks   |
|Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce|0              |General Grocery & Drinks   |
|Green Chile Anytime Sauce                                        |0              |General Grocery & Drinks   |
|Dry Nose Oil                                                     |0              |General Grocery & Dri